# Notebook 15 — Complete the ISTAT–CNR Patent Observatory maps

**Purpose:** take the dashboard ZIP provided by ChatGPT, copy the nine **already generated** Plotly map files from Google Drive (Notebook 14), and produce a single complete ZIP ready for **GitHub Pages**.

- **No embeddings are generated.**
- **No K-Means, HDBSCAN or UMAP is recomputed.**
- All 9 maps are based on the **same 2,400 patent IDs from the 6,400-patent pilot** (not new full-corpus projections).
- The full-corpus tables and metrics already included in the starter ZIP remain unchanged.

**Before running:** Download `istat_cnr_patent_observatory_starter.zip` from ChatGPT. Have access to the same Google Drive as Notebooks 12–14.

In [ ]:
# Import standard Python libraries: no package installation needed.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import json
import shutil
import re
import os

# Mount the Google Drive containing Notebook 14's outputs.
from google.colab import drive, files
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/istat_patents')
SOURCE_DIR = (ROOT / 'semantic_results_v11_clustering_robustness'
              / 'pilot_200_per_field' / 'all_texts' / 'interactive')

print('Expected map directory:', SOURCE_DIR)
print('Directory exists:', SOURCE_DIR.is_dir())

## 1. Provide the starter dashboard ZIP

Use the upload dialog to select `istat_cnr_patent_observatory_starter.zip`. Alternatively, place that file in `MyDrive/istat_patents/dashboard_publish/` and the next cell will find it automatically.

In [ ]:
# Use the ZIP from Google Drive if present, otherwise ask you to upload it.
zip_on_drive = ROOT / 'dashboard_publish' / 'istat_cnr_patent_observatory_starter.zip'

if zip_on_drive.is_file():
    starter_zip = zip_on_drive
    print('Using starter from Drive:', starter_zip)
else:
    print('Upload the STARTER ZIP downloaded from ChatGPT:')
    uploaded = files.upload()
    candidates = [Path('/content') / name for name in uploaded if name.lower().endswith('.zip')]
    if not candidates:
        raise FileNotFoundError('No ZIP uploaded. Upload the starter dashboard ZIP.')
    starter_zip = candidates[0]

# Work in a clean location so old maps do not get mixed with new files.
SITE = Path('/content/patent_observatory_site_complete')
if SITE.exists():
    shutil.rmtree(SITE)
SITE.mkdir(parents=True)

with ZipFile(starter_zip) as archive:
    # The supplied starter archive contains relative paths only.
    archive.extractall(SITE)
assert (SITE/'index.html').exists() and (SITE/'data'/'results.js').exists()
print('Unpacked starter dashboard:', SITE)

## 2. Import the nine UMAP HTML maps

The source notebook generated these files automatically. Copying them is fast and does not require an active Colab GPU. We also change the Plotly library reference to the locally bundled JavaScript so the maps work offline after upload.

In [ ]:
MODELS = [
    'bge-m3', 'qwen3-embedding-4b', 'qwen3-embedding-0.6b',
    'embeddinggemma-300m', 'paecter', 'pat-specter',
    'sbert-adapt-ub', 'mpnet', 'minilm09_chunked'
]

# Avoid silently claiming that a missing UMAP is available.
missing = []
added = []
for model in MODELS:
    filename = f'14_patent_map_{model}.html'
    source = SOURCE_DIR / filename
    if not source.is_file():
        missing.append(str(source))
        continue
    text = source.read_text(encoding='utf-8')
    # Check that the original file looks like an actual Plotly patent map.
    if 'Plotly.newPlot' not in text or 'patents.google.com' not in text:
        raise ValueError(f'Unexpected or incomplete map: {source}')
    # The starter already bundles Plotly. This change also makes offline browsing possible.
    text = re.sub(r'https://cdn\.plot\.ly/plotly-[^"\']+\.min\.js',
                  '../assets/plotly.min.js', text)
    # 2,400 points are manageable in SVG. Avoid WebGL-only rendering on restricted browsers.
    text = text.replace('"type":"scattergl"', '"type":"scatter"')
    (SITE/'maps'/filename).write_text(text,encoding='utf-8')
    added.append(model)
    print('Imported map:', model)

# This JavaScript manifest tells the dashboard which actual UMAPs it can display.
(SITE/'data'/'map_manifest.js').write_text(
    'window.AVAILABLE_MAPS=' + json.dumps(added) + ';\n', encoding='utf-8'
)
print(f'Imported {len(added)}/9 actual UMAP maps.')
if missing:
    print('Missing maps (no fake replacements will be generated):')
    for item in missing: print('  ',item)


## 3. Create the complete publication ZIP

This ZIP contains all static HTML/JS/CSS, full-corpus tables, pilot-method results, and the available maps. **Unzip it before uploading its contents to GitHub Pages.**

In [ ]:
# Save into the ISTAT patent project folder so the completed ZIP persists.
DEST = ROOT / 'dashboard_publish'
DEST.mkdir(parents=True, exist_ok=True)
output_zip = DEST / 'istat_cnr_patent_observatory_complete.zip'
with ZipFile(output_zip, 'w', ZIP_DEFLATED, compresslevel=6) as archive:
    for item in SITE.rglob('*'):
        if item.is_file():
            archive.write(item, arcname=item.relative_to(SITE))

print('Complete ZIP:', output_zip)
print('Size (MiB):',round(output_zip.stat().st_size/1024**2,1))
print('Maps included:',len(added))
print('Next: download this ZIP and follow DEPLOY_GITHUB_PAGES.md inside it.')
# Google Colab browser should offer a file download.
files.download(str(output_zip))

## 4. Publishing in a nutshell

1. Download and **unzip** the complete ZIP on your computer.
2. Create a **public** GitHub repository (for example `istat-cnr-patent-observatory`) and upload the **contents** of the unzipped folder, so `index.html` is at the repository root.
3. Open **Settings → Pages → Build and deployment → Deploy from a branch → main → /(root) → Save**.
4. Open the URL GitHub Pages displays. The charts and available maps are static: no running Colab, Streamlit or Python backend is needed.

**Research caveat:** Maps are from the 2,400-patent pilot projection. The dashboard's full-corpus metrics are separately labeled and refer to 63,968 patents.